In [0]:
# Définition des chemins et noms de tables pour la couche Silver
catalog = "bixi_mobility"

source_table = f"{catalog}.bronze.trips_raw"
target_table = f"{catalog}.silver.trips_clean"

checkpoint_location = f"/Volumes/{catalog}/silver/checkpoints/trips_clean"

In [0]:
# On lit la table Bronze en mode streaming, 
# Chaque nouvelle micro-batch Bronze sera automatiquement propagée vers le Silver
df_silver_source = spark.readStream.table(source_table)

In [0]:
from pyspark.sql.functions import col, from_unixtime, unix_timestamp, from_utc_timestamp

df_silver = (
    df_silver_source
    # Conversion epoch ms -> timestamp UTC, puis conversion au fuseau horaire de Montréal
    .withColumn("start_time", from_utc_timestamp((col("STARTTIMEMS") / 1000).cast("timestamp"), "America/Montreal"))
    .withColumn("end_time", from_utc_timestamp((col("ENDTIMEMS") / 1000).cast("timestamp"), "America/Montreal"))

    # Calcul de la durée du trajet en secondes, à partir des timestamps
    .withColumn(
        "duration_sec",
        (col("ENDTIMEMS") - col("STARTTIMEMS")) / 1000
    )

    # Renommage en snake_case pour plus de cohérence dans le pipeline
    .withColumnRenamed("STARTSTATIONNAME", "start_station_name")
    .withColumnRenamed("STARTSTATIONARRONDISSEMENT", "start_borough")
    .withColumnRenamed("STARTSTATIONLATITUDE", "start_lat")
    .withColumnRenamed("STARTSTATIONLONGITUDE", "start_lon")
    .withColumnRenamed("ENDSTATIONNAME", "end_station_name")
    .withColumnRenamed("ENDSTATIONARRONDISSEMENT", "end_borough")
    .withColumnRenamed("ENDSTATIONLATITUDE", "end_lat")
    .withColumnRenamed("ENDSTATIONLONGITUDE", "end_lon")

    # Filtre qualité : on écarte les trajets sans station ou avec une durée aberrante
    # (moins de 1 minute = probable erreur de badge, plus de 24h = anomalie système)
    .filter(col("start_station_name").isNotNull())
    .filter(col("end_station_name").isNotNull())
    .filter(col("duration_sec") >= 60)
    .filter(col("duration_sec") <= 86400)

    # On garde uniquement les colonnes utiles pour la suite du pipeline
    .select(
        "start_station_name", "start_borough", "start_lat", "start_lon",
        "end_station_name", "end_borough", "end_lat", "end_lon",
        "start_time", "end_time", "duration_sec"
    )
)

In [0]:
# Nettoyage du checkpoint corrompu suite aux tests précédents
dbutils.fs.rm(checkpoint_location, recurse=True)

# On s'assure aussi que la table cible est propre avant de repartir
spark.sql(f"DROP TABLE IF EXISTS {target_table}")

In [0]:
# Écriture en streaming vers la table Silver
# NB: trigger(availableNow=True) : traite uniquement les nouvelles données disponibles, puis s'arrête
(
    df_silver.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
SELECT start_time FROM bixi_mobility.silver.trips_clean LIMIT 5;